In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import polars as pl
import plotly.express as px
from utilsforecast.evaluation import evaluate
import plotly.io as pio

from utilsforecast.losses import *
from functools import partial
from plotting_utils import (
    plotly_series as plot_series,
)
import torch.nn as nn
import torch

In [3]:
import logging

from neuralforecast import NeuralForecast
from neuralforecast.models import (
    LSTM,
    NHITS,
    RNN,
    MLP,
    BiTCN,
    GRU,
    NBEATS,
    Autoformer,
    TFT,
    TCN,
    DeepAR,
    DLinear,
    TSMixer,
    PatchTST,
)
from neuralforecast.losses.pytorch import MAE

# logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

# Introduction to Deep Learning for Time Series Forecasting

Time series forecasting is the process of predicting future values based on previously observed data points, where the data is ordered in time. This is crucial in many real-world applications, such as:

- **Energy demand prediction** (like our smart meter dataset)
- **Stock price forecasting**
- **Weather prediction**
- **Sales forecasting**

## Why Deep Learning for Time Series?

Traditional statistical models (like ARIMA or Exponential Smoothing) have been widely used for time series forecasting. However, these models often struggle with:

- **Complex, non-linear patterns** in the data
- **Multiple seasonalities** or trends
- **Large-scale, high-dimensional datasets**

Deep learning models, especially neural networks, have shown great promise in overcoming these limitations by learning complex patterns directly from the data.

## What is Deep Learning?

Deep learning is a subset of machine learning that uses **artificial neural networks** with multiple layers (hence "deep") to model complex relationships. Each layer transforms the data, allowing the network to learn hierarchical representations.

### Analogy

Think of deep learning as a team of detectives, where each detective (layer) specializes in finding certain clues. The first detective looks for simple clues (like lines or shapes), the next combines those into more complex patterns, and so on. By the end, the team can solve very complex mysteries!

## Neural Networks for Time Series

The most common deep learning architectures for time series forecasting include:

- **Feedforward Neural Networks (FNNs):** Basic networks that can model simple relationships.
- **Recurrent Neural Networks (RNNs):** Designed to handle sequential data by maintaining a memory of previous inputs.
- **Long Short-Term Memory (LSTM) and Gated Recurrent Unit (GRU):** Special types of RNNs that can capture long-term dependencies.
- **Temporal Convolutional Networks (TCNs):** Use convolutional layers to model temporal patterns.
- **Transformer Models:** Recently, transformers have achieved state-of-the-art results in many sequence modeling tasks, including time series.

## How Does Deep Learning Work for Time Series?

At a high level, deep learning models for time series take a sequence of past observations as input and learn to predict future values. The model automatically learns which patterns and features are important, without the need for manual feature engineering.

### Example: Forecasting Energy Consumption

Suppose we have half-hourly energy consumption data from a smart meter. A deep learning model can learn:

- Daily and weekly usage patterns
- Effects of holidays or special events
- Sudden changes in behavior

## Mathematical Representation

Given a time series $\{y_t\}_{t=1}^T$, the goal is to predict future values $y_{T+1}, y_{T+2}, \ldots, y_{T+h}$ using past observations. A deep learning model learns a function $f$ such that:

$$
\hat{y}_{T+h} = f(y_T, y_{T-1}, \ldots, y_{T-p+1}; \theta)
$$

where:
- $p$ is the number of past observations used (the "window size")
- $\theta$ are the model parameters learned during training

## Key Advantages

- **Automatic feature extraction:** Learns relevant features from raw data
- **Handles non-linearity:** Captures complex relationships
- **Scalability:** Can process large datasets efficiently

## Common Beginner Questions

**Q: Do I need a lot of data for deep learning?**  
*A: Yes, deep learning models typically require more data than traditional models to perform well.*

**Q: Are deep learning models always better?**  
*A: Not always. For simple or small datasets, traditional models may outperform deep learning. It's important to compare different approaches.*

**Q: Is deep learning a "black box"?**  
*A: Deep learning models can be less interpretable, but there are tools and techniques to help understand their predictions.*

---

In the next sections, we'll explore how to prepare time series data for deep learning, build models using the `nixtla` library, and visualize results with `plotly`.

In [4]:
data = pl.read_parquet(
    "data/london_smart_meters/preprocessed/london_smart_meters_merged_block_0-7.parquet"
)
timestamp = data.group_by("LCLid").agg(
    pl.datetime_range(
        start=pl.col("start_timestamp"),
        end=pl.col("start_timestamp").dt.offset_by(
            pl.format("{}m", pl.col("series_length").sub(1).mul(30))
        ),
        interval="30m",
    ).alias("ds"),
)
data = timestamp.join(data, on="LCLid", how="inner").rename(
    {"LCLid": "unique_id", "energy_consumption": "y"}
)
data.head(5)

unique_id,ds,start_timestamp,frequency,y,series_length,stdorToU,Acorn,Acorn_grouped,file,holidays,visibility,windBearing,temperature,dewPoint,pressure,apparentTemperature,windSpeed,precipType,icon,humidity,summary,__index_level_0__
str,list[datetime[ns]],datetime[ns],str,list[f64],i64,str,str,str,str,list[str],list[f64],list[i64],list[f64],list[f64],list[f64],list[f64],list[f64],list[str],list[str],list[f64],list[str],i64
"""MAC000002""","[2012-10-13 00:00:00, 2012-10-13 00:30:00, … 2014-02-27 23:30:00]",2012-10-13 00:00:00,"""30min""","[0.263, 0.269, … 1.2180001]",24144,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[13.08, 13.08, … 14.03]","[186, 186, … 200]","[8.78, 8.78, … 3.93]","[6.28, 6.28, … 1.61]","[1007.7, 1007.7, … 1004.62]","[7.55, 7.55, … 1.42]","[2.28, 2.28, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""clear-night"", ""clear-night"", … ""clear-night""]","[0.84, 0.84, … 0.85]","[""Clear"", ""Clear"", … ""Clear""]",0
"""MAC000246""","[2012-01-01 00:00:00, 2012-01-01 00:30:00, … 2014-02-27 23:30:00]",2012-01-01 00:00:00,"""30min""","[0.509, 0.317, … 0.223]",37872,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[12.99, 12.99, … 14.03]","[229, 229, … 200]","[12.12, 12.12, … 3.93]","[10.97, 10.97, … 1.61]","[1008.1, 1008.1, … 1004.62]","[12.12, 12.12, … 1.42]","[5.9, 5.9, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""partly-cloudy-night"", ""partly-cloudy-night"", … ""clear-night""]","[0.93, 0.93, … 0.85]","[""Mostly Cloudy"", ""Mostly Cloudy"", … ""Clear""]",1
"""MAC000450""","[2012-03-23 00:00:00, 2012-03-23 00:30:00, … 2014-02-27 23:30:00]",2012-03-23 00:00:00,"""30min""","[1.337, 1.426, … null]",33936,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[3.19, 3.19, … 14.03]","[78, 78, … 200]","[8.76, 8.76, … 3.93]","[7.25, 7.25, … 1.61]","[1027.41, 1027.41, … 1004.62]","[7.59, 7.59, … 1.42]","[2.18, 2.18, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""fog"", ""fog"", … ""clear-night""]","[0.9, 0.9, … 0.85]","[""Foggy"", ""Foggy"", … ""Clear""]",2
"""MAC001074""","[2012-05-09 00:00:00, 2012-05-09 00:30:00, … 2014-02-27 23:30:00]",2012-05-09 00:00:00,"""30min""","[0.18, 0.086, … null]",31680,"""ToU""","""ACORN-""","""ACORN-""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[10.51, 10.51, … 14.03]","[215, 215, … 200]","[11.46, 11.46, … 3.93]","[10.23, 10.23, … 1.61]","[1007.39, 1007.39, … 1004.62]","[11.46, 11.46, … 1.42]","[2.35, 2.35, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""partly-cloudy-night"", ""partly-cloudy-night"", … ""clear-night""]","[0.92, 0.92, … 0.85]","[""Partly Cloudy"", ""Partly Cloudy"", … ""Clear""]",3
"""MAC003223""","[2012-09-18 00:00:00, 2012-09-18 00:30:00, … 2014-02-27 23:30:00]",2012-09-18 00:00:00,"""30min""","[0.076, 0.079, … 0.38]",25344,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[13.44, 13.44, … 14.03]","[236, 236, … 200]","[14.06, 14.06, … 3.93]","[10.82, 10.82, … 1.61]","[1011.09, 1011.09, … 1004.62]","[14.06, 14.06, … 1.42]","[3.86, 3.86, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""clear-night"", ""clear-night"", … ""clear-night""]","[0.81, 0.81, … 0.85]","[""Clear"", ""Clear"", … ""Clear""]",4


In [5]:
id_ = "unique_id"
time_ = "ds"
target_ = "y"
id_col = pl.col(id_)
time_col = pl.col(time_)
target_col = pl.col(target_)

In [6]:
data = (
    data.filter(pl.col("file").eq("block_7"))
    .select([time_, id_, target_])
    .explode([time_, target_])
)
data.head()

ds,unique_id,y
datetime[ns],str,f64
2012-01-01 00:00:00,"""MAC000050""",0.175
2012-01-01 00:30:00,"""MAC000050""",0.212
2012-01-01 01:00:00,"""MAC000050""",0.313
2012-01-01 01:30:00,"""MAC000050""",0.302
2012-01-01 02:00:00,"""MAC000050""",0.257


In [7]:
selected_id = "MAC000193"
data = data.filter(id_col.eq(selected_id)).with_columns(
    target_col.forward_fill().backward_fill()
)
data.head()

ds,unique_id,y
datetime[ns],str,f64
2012-01-01 00:00:00,"""MAC000193""",0.368
2012-01-01 00:30:00,"""MAC000193""",0.386
2012-01-01 01:00:00,"""MAC000193""",0.17
2012-01-01 01:30:00,"""MAC000193""",0.021
2012-01-01 02:00:00,"""MAC000193""",0.038


In [8]:
from functools import partial

metrics = [
    mae,
    mse,
    rmse,
    mape,
    smape,
    partial(mase, seasonality=48),
]

# Introduction to Recurrent Neural Networks (RNNs) for Electricity Load Demand Forecasting

## What is a Recurrent Neural Network (RNN)?

A **Recurrent Neural Network (RNN)** is a type of deep learning model specifically designed to handle sequential data, such as time series. Unlike traditional neural networks, RNNs have a unique architecture that allows them to "remember" information from previous steps in the sequence, making them especially powerful for tasks where the order of data matters.

### How Does an RNN Work?

- **Memory of the Past:**  
    At each time step $t$, an RNN takes the current input $x_t$ and combines it with information from the previous time step (the "hidden state" $h_{t-1}$). This allows the network to maintain a form of memory across the sequence.
- **Mathematical Representation:**  
    The hidden state is updated as follows:
    $$
    h_t = f(W_{xh} x_t + W_{hh} h_{t-1} + b_h)
    $$
    where:
    - $x_t$ is the input at time $t$
    - $h_{t-1}$ is the hidden state from the previous time step
    - $W_{xh}$ and $W_{hh}$ are weight matrices
    - $b_h$ is a bias term
    - $f$ is a non-linear activation function (like $\tanh$ or $\text{ReLU}$)

- **Output:**  
    The output at each time step can be computed as:
    $$
    y_t = g(W_{hy} h_t + b_y)
    $$
    where $g$ is typically a linear or non-linear function.

### Analogy

Imagine reading a book, one sentence at a time. To understand the current sentence, you need to remember what happened in previous sentences. RNNs mimic this process by carrying forward information from earlier in the sequence.

---

## Why Use RNNs for Electricity Load Demand Forecasting?

Electricity load demand is a classic time series problem: the amount of electricity consumed at any given time depends on previous consumption patterns, time of day, day of the week, season, and even special events.

### Key Reasons RNNs Are Suitable:

- **Capturing Temporal Dependencies:**  
    RNNs can learn how past electricity usage influences future demand, which is crucial for accurate forecasting.
- **Handling Variable-Length Sequences:**  
    RNNs can process sequences of different lengths, making them flexible for real-world data.
- **Modeling Complex Patterns:**  
    They can capture non-linear relationships and long-term dependencies that traditional models might miss.

---

## How Does an RNN Forecast Electricity Load?

1. **Input:**  
     The RNN receives a sequence of past electricity consumption values, e.g., $[y_{t-p+1}, \ldots, y_t]$.
2. **Processing:**  
     At each time step, the RNN updates its hidden state based on the current input and its memory of previous values.
3. **Prediction:**  
     The RNN outputs a forecast for the next time step(s), such as $y_{t+1}$, $y_{t+2}$, etc.

### Example

Suppose we want to forecast the next hour's electricity demand using the past 24 hours of data. The RNN will process the sequence of 24 hourly values and learn patterns such as daily cycles, spikes during certain hours, or drops at night.

---

## Mathematical Formulation

Given a time series $\{y_t\}_{t=1}^T$, the RNN learns a function $f$ such that:
$$
\hat{y}_{T+1} = f(y_T, y_{T-1}, \ldots, y_{T-p+1}; \theta)
$$
where:
- $p$ is the number of past observations (window size)
- $\theta$ are the model parameters learned during training

---

## Common Beginner Questions

**Q: Why not use a regular neural network?**  
*A: Regular (feedforward) neural networks treat each input independently and can't capture the sequential nature of time series data. RNNs are designed to handle sequences and remember past information.*

**Q: Can RNNs handle seasonality and trends?**  
*A: Yes, RNNs can learn seasonal patterns and trends if provided with enough data. However, for very long-term dependencies, advanced variants like LSTM or GRU are often used.*

**Q: Are RNNs hard to train?**  
*A: RNNs can be more challenging to train due to issues like vanishing gradients, but modern architectures and training techniques help address these problems.*

---

In the next section, we'll see how to implement an RNN for electricity load forecasting using the `nixtla` library and visualize the results with `plotly`.

In [9]:
# Minimal custom RNN model for one-step-ahead forecasting (NeuralForecast compatible)
import torch
import torch.nn as nn
from neuralforecast.common._base_model import BaseModel
from neuralforecast.common._modules import MLP
from neuralforecast.losses.pytorch import MAE
from typing import Optional


class CustomModel(BaseModel):
    EXOGENOUS_FUTR = False
    EXOGENOUS_HIST = False
    EXOGENOUS_STAT = False
    MULTIVARIATE = False
    RECURRENT = False  # Direct, not recursive

    def __init__(
        self,
        h: int,
        input_size: int = -1,
        inference_input_size: Optional[int] = None,
        futr_exog_list=None,
        hist_exog_list=None,
        stat_exog_list=None,
        exclude_insample_y=False,
        loss=MAE(),
        valid_loss=None,
        max_steps: int = 1000,
        learning_rate: float = 1e-3,
        num_lr_decays: int = -1,
        early_stop_patience_steps: int = -1,
        val_check_steps: int = 100,
        batch_size=32,
        valid_batch_size: Optional[int] = None,
        windows_batch_size=128,
        inference_windows_batch_size=1024,
        start_padding_enabled=False,
        step_size: int = 1,
        scaler_type: str = "robust",
        random_seed=1,
        drop_last_loader=False,
        alias: Optional[str] = None,
        optimizer=None,
        optimizer_kwargs=None,
        lr_scheduler=None,
        lr_scheduler_kwargs=None,
        dataloader_kwargs=None,
        **trainer_kwargs,
    ):
        super().__init__(
            h=h,
            input_size=input_size,
            inference_input_size=inference_input_size,
            futr_exog_list=futr_exog_list,
            hist_exog_list=hist_exog_list,
            stat_exog_list=stat_exog_list,
            exclude_insample_y=exclude_insample_y,
            loss=loss,
            valid_loss=valid_loss,
            max_steps=max_steps,
            learning_rate=learning_rate,
            num_lr_decays=num_lr_decays,
            early_stop_patience_steps=early_stop_patience_steps,
            val_check_steps=val_check_steps,
            batch_size=batch_size,
            valid_batch_size=valid_batch_size,
            windows_batch_size=windows_batch_size,
            inference_windows_batch_size=inference_windows_batch_size,
            start_padding_enabled=start_padding_enabled,
            step_size=step_size,
            scaler_type=scaler_type,
            random_seed=random_seed,
            drop_last_loader=drop_last_loader,
            alias=alias,
            optimizer=optimizer,
            optimizer_kwargs=optimizer_kwargs,
            lr_scheduler=lr_scheduler,
            lr_scheduler_kwargs=lr_scheduler_kwargs,
            dataloader_kwargs=dataloader_kwargs,
            **trainer_kwargs,
        )

    def forward(self, windows_batch):
        pass

In [ ]:
m = nn.Linear(20, 30)
input = torch.randn(128, 20)
output = m(input)
print(output.size())

In [84]:
class CustomRNN(CustomModel):
    EXOGENOUS_FUTR = False
    EXOGENOUS_HIST = False
    EXOGENOUS_STAT = False
    MULTIVARIATE = False
    RECURRENT = False

    def __init__(
        self,
        h: int,
        input_size: int = -1,
        encoder_hidden_size: int = 128,
        encoder_n_layers: int = 2,
        decoder_n_layers: int = 2,
        **kwargs,
    ):
        super().__init__(
            h=h,
            input_size=input_size,
            **kwargs,
        )
        self.example_input_array = (
            {"insample_y": torch.Tensor(self.batch_size, input_size, 1)},
        )

        self.encoder_hidden_size = encoder_hidden_size
        self.encoder_n_layers = encoder_n_layers

        self.rnn = nn.RNN(
            input_size=1,
            hidden_size=encoder_hidden_size,
            num_layers=encoder_n_layers,
            batch_first=True,
        )

        # Decoder MLP: Stack of Linear layers with ReLU activations in between and dropout=0.0
        decoder_layers = []
        in_features = self.encoder_hidden_size
        for i in range(decoder_n_layers - 1):
            decoder_layers.append(nn.Linear(in_features, self.encoder_hidden_size))
            decoder_layers.append(nn.ReLU())
            decoder_layers.append(nn.Dropout(0.0))
            in_features = self.encoder_hidden_size
        decoder_layers.append(nn.Linear(in_features, self.loss.outputsize_multiplier))
        self.mlp_decoder = nn.Sequential(*decoder_layers)

    def forward(self, windows_batch):
        encoder_input = windows_batch["insample_y"]

        hidden_state, _ = self.rnn(
            encoder_input, None
        )  # [b, seq_len, rnn_hidden_state]
        hidden_state = hidden_state[
            :, -self.h :
        ]  # [b, seq_len, rnn_hidden_state] -> [b, h, rnn_hidden_state]

        output = self.mlp_decoder(
            hidden_state
        )  # [b, h, rnn_hidden_state + f] -> [b, seq_len, n_output]

        return output[:, -self.h :]


In [86]:
horizon = 48

# Try different hyperparmeters to improve accuracy.
models = [
    CustomRNN(
        input_size=2 * horizon,
        h=horizon,  # Forecast horizon
        max_steps=100,  # Number of steps to train
        scaler_type="standard",  # Type of scaler to normalize data
        encoder_hidden_size=64,  # Defines the size of the hidden state of the LSTM
        encoder_n_layers=2,  # Number of layers in the RNN
        val_check_steps=10,
        alias="CustomRNN_daily",  # Alias for the model
    ),
    CustomRNN(
        input_size=2 * horizon * 7,
        h=horizon,  # Forecast horizon
        max_steps=100,  # Number of steps to train
        scaler_type="standard",  # Type of scaler to normalize data
        encoder_hidden_size=64,  # Defines the size of the hidden state of the LSTM
        encoder_n_layers=2,  # Number of layers in the RNN
        val_check_steps=10,
        alias="CustomRNN_weekly",  # Alias for the model
    ),
]

nf = NeuralForecast(models=models, freq="30m")
y_hat = nf.cross_validation(
    data,
    step_size=48,
    n_windows=1,
    val_size=96,
).drop("cutoff")

Seed set to 1
Seed set to 1
Seed set to 1
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name         | Type          | Params | Mode  | In sizes           | Out sizes                  
----------------------------------------------------------------------------------------------------------
0 | loss         | MAE           | 0      | train | ?                  | ?                          
1 | padder_train | ConstantPad1d | 0      | train | ?                  | ?                          
2 | scaler       | TemporalNorm  | 0      | train | ?                  | ?                          
3 | rnn          | RNN           | 12.6 K | train | [[32, 96, 1], '?'] | [[32, 96, 64], [2, 32, 64]]
4 | mlp_decoder  | Sequential    | 4.2 K  | train | [32, 48, 64]       | [32, 48, 1]                
--------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=100` reached.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Predicting: |          | 0/? [00:00<?, ?it/s]

In [87]:
fig = plot_series(y_hat, y_hat)
fig.show()

evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

unique_id,metric,CustomRNN_daily,CustomRNN_weekly
str,str,f64,f64
"""MAC000193""","""mae""",0.275803,0.271373
"""MAC000193""","""mse""",0.267904,0.268064
"""MAC000193""","""rmse""",0.517595,0.517749
"""MAC000193""","""mape""",3.329773,3.158282
"""MAC000193""","""smape""",0.376013,0.335451
"""MAC000193""","""mase""",1.596216,1.570581


# Introduction to Long Short-Term Memory (LSTM) Networks

## What is an LSTM?

A **Long Short-Term Memory (LSTM)** network is a special type of Recurrent Neural Network (RNN) designed to better capture long-term dependencies in sequential data, such as time series. LSTMs are widely used in deep learning for tasks where remembering information over long sequences is crucial.

---

## The Problem with Standard RNNs

While RNNs are powerful for modeling sequences, they suffer from a major limitation called the **vanishing gradient problem**. This means that as the network tries to learn relationships over long sequences, the influence of earlier data points fades away during training. As a result, standard RNNs struggle to remember information from the distant past.

**Analogy:**  
Imagine trying to recall what you had for breakfast a month ago versus what you ate this morning. Standard RNNs tend to "forget" older information, just like our memory fades over time.

---

## How LSTMs Solve the RNN Problem

LSTMs introduce a clever architecture with **memory cells** and **gates** that control the flow of information. This allows them to:

- **Remember important information for long periods**
- **Forget irrelevant details**
- **Update their memory as new data arrives**

### Key Components of an LSTM Cell

- **Cell State ($C_t$):**  
    Acts as a conveyor belt, carrying information across time steps with minimal changes.
- **Gates:**  
    Special neural network layers that decide what information to keep, forget, or add:
    - **Forget Gate ($f_t$):** Decides what to discard from the cell state.
    - **Input Gate ($i_t$):** Decides what new information to store.
    - **Output Gate ($o_t$):** Decides what to output from the cell.

### Mathematical Representation

At each time step $t$, the LSTM updates its memory using the following equations:

$$
\begin{align*}
f_t &= \sigma(W_f \cdot [h_{t-1}, x_t] + b_f) \\
i_t &= \sigma(W_i \cdot [h_{t-1}, x_t] + b_i) \\
\tilde{C}_t &= \tanh(W_C \cdot [h_{t-1}, x_t] + b_C) \\
C_t &= f_t * C_{t-1} + i_t * \tilde{C}_t \\
o_t &= \sigma(W_o \cdot [h_{t-1}, x_t] + b_o) \\
h_t &= o_t * \tanh(C_t)
\end{align*}
$$

Where:
- $x_t$ is the input at time $t$
- $h_{t-1}$ is the previous hidden state
- $C_{t-1}$ is the previous cell state
- $\sigma$ is the sigmoid activation function
- $*$ denotes element-wise multiplication

**In simple terms:**  
LSTMs learn what to remember, what to forget, and what to output at each step, making them much better at capturing long-term patterns.

---

## Why Are LSTMs Useful for Electricity Load Forecasting?

Electricity load data often contains:

- **Daily and weekly cycles** (seasonality)
- **Long-term trends**
- **Sudden changes** (e.g., holidays, weather events)

LSTMs are especially effective for this kind of data because:

- **They can learn both short-term and long-term dependencies** (e.g., yesterday’s usage and last week’s pattern)
- **They handle noisy and complex patterns** better than traditional models
- **They require less manual feature engineering**—the model learns relevant features automatically

### Real-World Example

Suppose you want to forecast electricity demand for the next day. The LSTM can learn:

- The typical daily usage pattern (e.g., higher in the evening)
- Weekly cycles (e.g., lower usage on weekends)
- Effects of special events (e.g., a holiday spike)

---

## Common Beginner Questions

**Q: Are LSTMs always better than RNNs?**  
*A: For most real-world time series with long-term dependencies, LSTMs outperform standard RNNs. However, for very short sequences, the difference may be small.*

**Q: Do LSTMs need a lot of data?**  
*A: Like most deep learning models, LSTMs perform best with larger datasets, but they are more robust to sequence length than standard RNNs.*

**Q: Are LSTMs hard to train?**  
*A: LSTMs are more complex than basic RNNs, but modern libraries (like `nixtla.neuralforecast`) make them easy to use in practice.*

---

In the next section, we’ll see how to implement an LSTM for electricity load forecasting using the `nixtla` library and visualize the results with `plotly`.

In [88]:
class CustomLSTM(CustomModel):
    EXOGENOUS_FUTR = False
    EXOGENOUS_HIST = False
    EXOGENOUS_STAT = False
    MULTIVARIATE = False
    RECURRENT = False

    def __init__(
        self,
        h: int,
        input_size: int = -1,
        encoder_hidden_size: int = 128,
        encoder_n_layers: int = 2,
        decoder_n_layers: int = 2,
        **kwargs,
    ):
        super().__init__(
            h=h,
            input_size=input_size,
            **kwargs,
        )
        self.example_input_array = (
            {"insample_y": torch.Tensor(self.batch_size, input_size, 1)},
        )

        self.encoder_hidden_size = encoder_hidden_size
        self.encoder_n_layers = encoder_n_layers

        self.encoder = nn.LSTM(
            input_size=1,
            hidden_size=encoder_hidden_size,
            num_layers=encoder_n_layers,
            batch_first=True,
        )

        # Decoder MLP: Stack of Linear layers with ReLU activations in between and dropout=0.0
        decoder_layers = []
        in_features = self.encoder_hidden_size
        for i in range(decoder_n_layers - 1):
            decoder_layers.append(nn.Linear(in_features, self.encoder_hidden_size))
            decoder_layers.append(nn.ReLU())
            decoder_layers.append(nn.Dropout(0.0))
            in_features = self.encoder_hidden_size
        decoder_layers.append(nn.Linear(in_features, self.loss.outputsize_multiplier))
        self.mlp_decoder = nn.Sequential(*decoder_layers)

    def forward(self, windows_batch):
        encoder_input = windows_batch["insample_y"]

        hidden_state, _ = self.encoder(
            encoder_input, None
        )  # [b, seq_len, rnn_hidden_state]
        hidden_state = hidden_state[
            :, -self.h :
        ]  # [b, seq_len, rnn_hidden_state] -> [b, h, rnn_hidden_state]

        output = self.mlp_decoder(
            hidden_state
        )  # [b, h, rnn_hidden_state + f] -> [b, seq_len, n_output]

        return output[:, -self.h :]


In [89]:
horizon = 48

# Try different hyperparmeters to improve accuracy.
models = [
    CustomLSTM(
        input_size=2 * horizon,
        h=horizon,  # Forecast horizon
        max_steps=100,  # Number of steps to train
        scaler_type="standard",  # Type of scaler to normalize data
        encoder_hidden_size=64,  # Defines the size of the hidden state of the LSTM
        encoder_n_layers=2,  # Number of layers in the RNN
        val_check_steps=10,
    ),
]

nf = NeuralForecast(models=models, freq="30m")
y_hat = nf.cross_validation(
    data,
    step_size=48,
    n_windows=1,
    val_size=96,
).drop("cutoff")

Seed set to 1
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name         | Type          | Params | Mode  | In sizes           | Out sizes                                 
-------------------------------------------------------------------------------------------------------------------------
0 | loss         | MAE           | 0      | train | ?                  | ?                                         
1 | padder_train | ConstantPad1d | 0      | train | ?                  | ?                                         
2 | scaler       | TemporalNorm  | 0      | train | ?                  | ?                                         
3 | encoder      | LSTM          | 50.4 K | train | [[32, 96, 1], '?'] | [[32, 96, 64], [[2, 32, 64], [2, 32, 64]]]
4 | mlp_decoder  | Sequential    | 4.2 K  | train | [

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=100` reached.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Predicting: |          | 0/? [00:00<?, ?it/s]

In [90]:
fig = plot_series(y_hat, y_hat)
fig.show()

evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

unique_id,metric,CustomLSTM
str,str,f64
"""MAC000193""","""mae""",0.275391
"""MAC000193""","""mse""",0.250773
"""MAC000193""","""rmse""",0.500773
"""MAC000193""","""mape""",3.411916
"""MAC000193""","""smape""",0.388627
"""MAC000193""","""mase""",1.593831


# Introduction to Gated Recurrent Units (GRUs) for Electricity Load Forecasting

## What is a GRU?

A **Gated Recurrent Unit (GRU)** is a type of recurrent neural network (RNN) architecture designed to model sequential data, such as time series. GRUs were introduced as a simpler and more efficient alternative to Long Short-Term Memory (LSTM) networks, while still addressing the same challenges of learning long-term dependencies.

---

## Why Were GRUs Developed?

While LSTMs are powerful for capturing long-term patterns, they have a complex structure with multiple gates and a separate cell state. This complexity can make LSTMs slower to train and more computationally expensive, especially for large datasets or real-time applications.

**GRUs simplify the LSTM architecture** by combining some of the gates and removing the separate cell state, making them:

- Faster to train
- Easier to implement
- Often just as effective for many time series tasks

---

## How Does a GRU Work?

A GRU has two main gates:

- **Update Gate ($z_t$):** Decides how much of the past information to keep.
- **Reset Gate ($r_t$):** Decides how much of the past information to forget.

At each time step $t$, the GRU updates its hidden state $h_t$ using the following equations:

$$
\begin{align*}
z_t &= \sigma(W_z x_t + U_z h_{t-1} + b_z) \\
r_t &= \sigma(W_r x_t + U_r h_{t-1} + b_r) \\
\tilde{h}_t &= \tanh(W_h x_t + U_h (r_t * h_{t-1}) + b_h) \\
h_t &= (1 - z_t) * h_{t-1} + z_t * \tilde{h}_t
\end{align*}
$$

Where:
- $x_t$ is the input at time $t$
- $h_{t-1}$ is the previous hidden state
- $\sigma$ is the sigmoid activation function
- $*$ denotes element-wise multiplication

**Key differences from LSTM:**
- GRUs merge the cell state and hidden state into a single vector.
- They use fewer gates (just update and reset), making them computationally lighter.

---

## Why Use GRUs for Electricity Load Forecasting?

Electricity load forecasting involves predicting future energy consumption based on past usage patterns. This data often contains:

- **Short-term and long-term dependencies** (e.g., daily and weekly cycles)
- **Sudden changes** (e.g., holidays, weather events)

GRUs are well-suited for this task because:

- **They efficiently capture both short- and long-term patterns** in the data.
- **They train faster than LSTMs**, which is useful for large smart meter datasets.
- **They require fewer parameters**, reducing the risk of overfitting on smaller datasets.

---

## Analogy

Imagine a GRU as a smart assistant that decides, at each moment, what information from the past is important to remember and what can be safely forgotten. This helps it focus on the most relevant patterns for making accurate forecasts.

---

## Common Beginner Questions

**Q: Are GRUs always better than LSTMs?**  
*A: Not always. GRUs are simpler and often perform just as well as LSTMs, but for some very complex or long sequences, LSTMs may have a slight edge. It's best to try both and compare.*

**Q: When should I use a GRU instead of an LSTM?**  
*A: Use GRUs when you want faster training, have limited computational resources, or your dataset is not extremely large or complex.*

**Q: Do GRUs need as much data as LSTMs?**  
*A: GRUs have fewer parameters, so they can sometimes perform better than LSTMs on smaller datasets.*

---

## Summary

- **GRUs are a streamlined alternative to LSTMs** for modeling sequential data like electricity load.
- **They use fewer gates and no separate cell state**, making them faster and easier to train.
- **For many time series forecasting tasks, including electricity demand, GRUs offer similar or better performance with less complexity.**

---

In the next section, we'll see how to implement a GRU for electricity load forecasting using the `nixtla` library and visualize the results with `plotly`.

In [93]:
class CustomGRU(CustomModel):
    EXOGENOUS_FUTR = False
    EXOGENOUS_HIST = False
    EXOGENOUS_STAT = False
    MULTIVARIATE = False
    RECURRENT = False

    def __init__(
        self,
        h: int,
        input_size: int = -1,
        encoder_hidden_size: int = 128,
        encoder_n_layers: int = 2,
        decoder_n_layers: int = 2,
        **kwargs,
    ):
        super().__init__(
            h=h,
            input_size=input_size,
            **kwargs,
        )
        self.example_input_array = (
            {"insample_y": torch.Tensor(self.batch_size, input_size, 1)},
        )

        self.encoder_hidden_size = encoder_hidden_size
        self.encoder_n_layers = encoder_n_layers

        self.encoder = nn.GRU(
            input_size=1,
            hidden_size=encoder_hidden_size,
            num_layers=encoder_n_layers,
            batch_first=True,
        )

        # Decoder MLP: Stack of Linear layers with ReLU activations in between and dropout=0.0
        decoder_layers = []
        in_features = self.encoder_hidden_size
        for i in range(decoder_n_layers - 1):
            decoder_layers.append(nn.Linear(in_features, self.encoder_hidden_size))
            decoder_layers.append(nn.ReLU())
            decoder_layers.append(nn.Dropout(0.0))
            in_features = self.encoder_hidden_size
        decoder_layers.append(nn.Linear(in_features, self.loss.outputsize_multiplier))
        self.mlp_decoder = nn.Sequential(*decoder_layers)

    def forward(self, windows_batch):
        encoder_input = windows_batch["insample_y"]

        hidden_state, _ = self.encoder(
            encoder_input, None
        )  # [b, seq_len, rnn_hidden_state]
        hidden_state = hidden_state[
            :, -self.h :
        ]  # [b, seq_len, rnn_hidden_state] -> [b, h, rnn_hidden_state]

        output = self.mlp_decoder(
            hidden_state
        )  # [b, h, rnn_hidden_state + f] -> [b, seq_len, n_output]

        return output[:, -self.h :]


In [94]:
horizon = 48

# Try different hyperparmeters to improve accuracy.
models = [
    CustomGRU(
        input_size=2 * horizon,
        h=horizon,  # Forecast horizon
        max_steps=100,  # Number of steps to train
        scaler_type="standard",  # Type of scaler to normalize data
        encoder_hidden_size=64,  # Defines the size of the hidden state of the LSTM
        encoder_n_layers=2,  # Number of layers in the RNN
        val_check_steps=10,
    ),
]

nf = NeuralForecast(models=models, freq="30m")
y_hat = nf.cross_validation(
    data,
    step_size=48,
    n_windows=1,
    val_size=96,
).drop("cutoff")

Seed set to 1
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name         | Type          | Params | Mode  | In sizes           | Out sizes                  
----------------------------------------------------------------------------------------------------------
0 | loss         | MAE           | 0      | train | ?                  | ?                          
1 | padder_train | ConstantPad1d | 0      | train | ?                  | ?                          
2 | scaler       | TemporalNorm  | 0      | train | ?                  | ?                          
3 | encoder      | GRU           | 37.8 K | train | [[32, 96, 1], '?'] | [[32, 96, 64], [2, 32, 64]]
4 | mlp_decoder  | Sequential    | 4.2 K  | train | [32, 48, 64]       | [32, 48, 1]                
------------------------------------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=100` reached.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
HPU available: False, using: 0 HPUs


Predicting: |          | 0/? [00:00<?, ?it/s]

In [95]:
fig = plot_series(y_hat, y_hat)
fig.show()

evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

unique_id,metric,CustomGRU
str,str,f64
"""MAC000193""","""mae""",0.280345
"""MAC000193""","""mse""",0.273387
"""MAC000193""","""rmse""",0.522864
"""MAC000193""","""mape""",3.381879
"""MAC000193""","""smape""",0.368681
"""MAC000193""","""mase""",1.622505


# LSTM-to-LSTM Sequence-to-Sequence Models for Time Series Forecasting

## What is an LSTM-to-LSTM Model?

So far, we've used a **sequence-to-sequence (seq2seq)** architecture where an RNN, LSTM, or GRU encodes the input sequence, and a simple Multi-Layer Perceptron (MLP) decodes the output. While this approach works well for many tasks, it can be limiting when the output sequence is long or when we want the decoder to have its own memory of past predictions.

An **LSTM-to-LSTM** model is a seq2seq architecture where **both the encoder and decoder are LSTM networks**. This design is inspired by advances in natural language processing (like machine translation), but it is also very powerful for time series forecasting.

---

## Why Use LSTM as Both Encoder and Decoder?

- **Richer Memory:**  
    Both the encoder and decoder can learn long-term dependencies, allowing the model to capture complex temporal patterns in both the input and output sequences.
- **Flexible Forecasting:**  
    The decoder LSTM can generate multi-step forecasts, using its own previous outputs as inputs for future predictions (a process called "autoregression").
- **Handling Variable-Length Sequences:**  
    LSTM-to-LSTM models can handle input and output sequences of different lengths, which is useful for many real-world forecasting problems.

---

## How Does LSTM-to-LSTM Work?

1. **Encoder LSTM:**  
     Processes the input sequence (e.g., past energy consumption) and summarizes it into a context vector (the final hidden and cell states).
2. **Decoder LSTM:**  
     Starts from the encoder's context and generates the forecasted sequence, one step at a time. At each step, the decoder can use its previous prediction as input for the next time step.

### Mathematical Representation

Suppose we have an input sequence $[y_{t-p+1}, \ldots, y_t]$ and want to forecast $[y_{t+1}, \ldots, y_{t+h}]$.

- **Encoder:**  
    Processes the input sequence and outputs a context $(h_t, C_t)$.
- **Decoder:**  
    At each future step $k$, predicts $\hat{y}_{t+k}$ using the decoder LSTM:
    $$
    (h_{t+k}, C_{t+k}) = \text{LSTM}_{\text{dec}}(\hat{y}_{t+k-1}, (h_{t+k-1}, C_{t+k-1}))
    $$
    $$
    \hat{y}_{t+k} = \text{MLP}(h_{t+k})
    $$
    where $\hat{y}_{t}$ is initialized with the last value of the input sequence or a special start token.

---

## Analogy

Think of the encoder LSTM as a skilled listener who summarizes a long story (the past data), and the decoder LSTM as a storyteller who uses that summary to generate a new story (the forecast), remembering what it has already said.

---

## Key Advantages

- **Captures complex, long-term dependencies** in both input and output sequences.
- **Improves multi-step forecasting** by allowing the decoder to model dependencies between future time steps.
- **Widely used in state-of-the-art forecasting models** for energy, weather, and more.

---

## Common Beginner Questions

**Q: Why not just use an MLP as the decoder?**  
*A: An MLP decoder treats each forecasted step independently, while an LSTM decoder can model dependencies between future steps, leading to more realistic and accurate forecasts.*

**Q: Is training LSTM-to-LSTM harder?**  
*A: It can be more computationally intensive, but modern libraries like `nixtla.neuralforecast` make implementation straightforward.*

**Q: Can I use teacher forcing with LSTM decoders?**  
*A: Yes! During training, you can feed the true previous value to the decoder (teacher forcing), which helps the model learn faster and more accurately.*

---

In the next section, we'll see how to implement an LSTM-to-LSTM seq2seq model for electricity load forecasting using the `nixtla` library, and visualize the results with `plotly`.

In [126]:
import torch
import torch.nn as nn


class EncoderLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)

    def forward(self, x):
        # x: [batch_size, seq_len, input_dim]
        outputs, (hidden, cell) = self.lstm(x)
        return hidden, cell


In [127]:
class DecoderLSTM(nn.Module):
    def __init__(self, output_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(output_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, y_prev, hidden, cell):
        # y_prev: [batch_size, 1, output_dim]
        output, (hidden, cell) = self.lstm(y_prev, (hidden, cell))
        prediction = self.fc(output)  # [batch_size, 1, output_dim]
        return prediction, hidden, cell



## Key Points

- The **encoder** summarizes the input sequence into hidden states.
- The **decoder** generates each forecast step, using its previous output as input.
- This approach is flexible and can be extended to multivariate or exogenous inputs.

---

**Tip:**  
For improved training, you can use "teacher forcing" (feeding the true previous value to the decoder during training), but for simplicity, this example uses only the model's own predictions as decoder inputs.

In [ ]:
class CustomLSTM2LSTM(CustomModel):
    EXOGENOUS_FUTR = False
    EXOGENOUS_HIST = False
    EXOGENOUS_STAT = False
    MULTIVARIATE = False
    RECURRENT = False

    def __init__(
        self,
        h: int,
        input_size: int = -1,
        input_dim: int = 1,
        hidden_dim: int = 128,
        output_dim: int = 1,
        num_layers: int = 2,
        teacher_forcing_rate: float = 0.5,
        **kwargs,
    ):
        super().__init__(
            h=h,
            input_size=input_size,
            **kwargs,
        )
        self.example_input_array = (
            {"insample_y": torch.Tensor(self.batch_size, input_size, 1)},
        )
        self.teacher_forcing_rate = teacher_forcing_rate

        self.encoder = EncoderLSTM(input_dim, hidden_dim, num_layers)
        self.decoder = DecoderLSTM(output_dim, hidden_dim, num_layers)

    def training_step(self, batch, batch_idx):
        # Set horizon to h_train in case of recurrent model to speed up training
        if self.RECURRENT:
            self.h = self.h_train

        # windows: [Ws, L + h, C, n_series] or [Ws, L + h, C]
        y_idx = batch["y_idx"]

        windows = self._create_windows(batch, step="train")
        original_outsample_y = torch.clone(
            windows["temporal"][:, self.input_size :, y_idx]
        )
        windows = self._normalization(windows=windows, y_idx=y_idx)

        # Parse windows
        (
            insample_y,
            insample_mask,
            outsample_y,
            outsample_mask,
            hist_exog,
            futr_exog,
            stat_exog,
        ) = self._parse_windows(batch, windows)

        windows_batch = dict(
            insample_y=insample_y,  # [Ws, L, n_series]
            insample_mask=insample_mask,  # [Ws, L, n_series]
            outsample_y=outsample_y,  # [Ws, h, n_series]
            futr_exog=futr_exog,  # univariate: [Ws, L, F]; multivariate: [Ws, F, L, n_series]
            hist_exog=hist_exog,  # univariate: [Ws, L, X]; multivariate: [Ws, X, L, n_series]
            stat_exog=stat_exog,
        )  # univariate: [Ws, S]; multivariate: [n_series, S]

        # Model Predictions
        output = self(windows_batch)
        output = self.loss.domain_map(output)

        if self.loss.is_distribution_output:
            y_loc, y_scale = self._get_loc_scale(y_idx)
            outsample_y = original_outsample_y
            distr_args = self.loss.scale_decouple(
                output=output, loc=y_loc, scale=y_scale
            )
            loss = self.loss(y=outsample_y, distr_args=distr_args, mask=outsample_mask)
        else:
            loss = self.loss(
                y=outsample_y, y_hat=output, y_insample=insample_y, mask=outsample_mask
            )

        if torch.isnan(loss):
            print("Model Parameters", self.hparams)
            print("insample_y", torch.isnan(insample_y).sum())
            print("outsample_y", torch.isnan(outsample_y).sum())
            raise Exception("Loss is NaN, training stopped.")

        train_loss_log = loss.detach().item()
        self.log(
            "train_loss",
            train_loss_log,
            batch_size=outsample_y.size(0),
            prog_bar=True,
            on_epoch=True,
        )
        self.train_trajectories.append((self.global_step, train_loss_log))

        self.h = self.horizon_backup

        return loss

    def forward(self, windows_batch):
        encoder_input = windows_batch[
            "insample_y"
        ]  # [batch_size, src_seq_len, input_dim]
        outsample_y = windows_batch.get(
            "outsample_y", None
        )  # [batch_size, h, output_dim]

        outputs = []

        # Encode the input sequence
        hidden, cell = self.encoder(encoder_input)

        # First decoder input: last value of input sequence
        decoder_input = encoder_input[:, -1:, :]  # [batch_size, 1, input_dim]

        for t in range(self.h):
            out, hidden, cell = self.decoder(decoder_input, hidden, cell)
            outputs.append(out)
            # Teacher forcing: use the true value as next input during training
            # Teacher forcing: with probability self.teacher_forcing_rate, use the true value as next input
            if (
                self.training
                and outsample_y is not None
                and torch.rand(1).item() < self.teacher_forcing_rate
            ):
                decoder_input = outsample_y[:, t : t + 1, :]
            else:
                decoder_input = out  # Use own prediction as next input

        outputs = torch.cat(outputs, dim=1)  # [batch_size, tgt_len, output_dim]
        return outputs


In [ ]:
horizon = 48

# Try different hyperparmeters to improve accuracy.
models = [
    CustomLSTM2LSTM(
        input_size=2 * horizon,
        h=horizon,  # Forecast horizon
        max_steps=100,  # Number of steps to train
        scaler_type="standard",  # Type of scaler to normalize data
        val_check_steps=10,
        input_dim=1,
        hidden_dim=64,  # Defines the size of the hidden state of the LSTM
        output_dim=1,  # Output dimension of the decoder
        num_layers=2,  # Number of layers in the LSTM
        teacher_forcing_rate=0,
        alias="CustomLSTM2LSTM",  # Alias for the model
    ),
    CustomLSTM2LSTM(
        input_size=2 * horizon,
        h=horizon,  # Forecast horizon
        max_steps=100,  # Number of steps to train
        scaler_type="standard",  # Type of scaler to normalize data
        val_check_steps=10,
        input_dim=1,
        hidden_dim=64,  # Defines the size of the hidden state of the LSTM
        output_dim=1,  # Output dimension of the decoder
        num_layers=2,  # Number of layers in the LSTM
        teacher_forcing_rate=0.5,
        alias="CustomLSTM2LSTM_tf",  # Alias for the model
    ),
]

nf = NeuralForecast(models=models, freq="30m")
y_hat = nf.cross_validation(
    data,
    step_size=48,
    n_windows=1,
    val_size=96,
).drop("cutoff")

Seed set to 1
Seed set to 1
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name         | Type          | Params | Mode  | In sizes                               | Out sizes                             
-----------------------------------------------------------------------------------------------------------------------------------------
0 | loss         | MAE           | 0      | train | ?                                      | ?                                     
1 | padder_train | ConstantPad1d | 0      | train | ?                                      | ?                                     
2 | scaler       | TemporalNorm  | 0      | train | ?                                      | ?                                     
3 | encoder      | EncoderLSTM   | 50.4 K | train | [32, 96, 1]                            | [[2, 32, 64], [2, 32, 64]]            
4 | decoder      | DecoderLSTM   | 50.5 K | train | [[32, 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name         | Type          | Params | Mode  | In sizes                               | Out sizes                             
-----------------------------------------------------------------------------------------------------------------------------------------
0 | loss         | MAE           | 0      | train | ?                                      | ?                                     
1 | padder_train | ConstantPad1d | 0      | train | ?                                      | ?                                     
2 | scaler       | TemporalNorm  | 0      | train | ?                                      | ?                                     
3 | encoder      | EncoderLSTM   | 50.4 K | train | [32, 96, 1]                            | [[2, 32, 64], [2, 32, 64]]            
4 | decoder      | DecoderLSTM   | 50.5 K | train | [[32, 1, 1], [2, 32, 64], [2, 32, 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

RuntimeError: For unbatched 2-D input, hx and cx should also be 2-D but got (3-D, 3-D) tensors

: 

In [140]:
fig = plot_series(y_hat, y_hat)
fig.show()

evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

unique_id,metric,CustomLSTM2LSTM,CustomLSTM2LSTM_tf
str,str,f64,f64
"""MAC000193""","""mae""",0.329701,0.341429
"""MAC000193""","""mse""",0.298356,0.295493
"""MAC000193""","""rmse""",0.54622,0.543592
"""MAC000193""","""mape""",4.865662,5.937963
"""MAC000193""","""smape""",0.549825,0.547456
"""MAC000193""","""mase""",1.908151,1.976032
